In [1]:
from pathlib import Path
from typing import Union
import pandas as pd
import numpy as np
from transformers import pipeline
import os
import re

/home/maciej/anaconda3/envs/twitter-posting/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mapping_pos = {
    "Prompt violating policy": np.nan,
    "Extremely high intensity of positive emotional affect": 5,
    "High intensity of positive emotional affect": 4,
    "Average intensity of positive emotional affect": 3,
    "Low intensity of positive emotional affect": 2,
    "Extremely low intensity of positive emotional affect": 1
}
mapping_neg = {
    "Prompt violating policy": np.nan,
    "Extremely high intensity of negative emotional affect": 5,
    "High intensity of negative emotional affect": 4,
    "Average intensity of negative emotional affect": 3,
    "Low intensity of negative emotional affect": 2,
    "Extremely low intensity of negative emotional affect": 1
}

mapping_int = {
    "Prompt violating policy": np.nan,
    "Extremely high intelligence": 5,
    "High intelligence": 4,
    "Average intelligence": 3,
    "Low intelligence": 2,
    "Extremely low intelligence": 1
}

In [3]:
int_path = Path("../data/reporting/intelligence")
ea_path = Path("../data/reporting/emotional_affect")

In [4]:
files_int = os.listdir(int_path)
files_ea = os.listdir(ea_path)
files = []
for file in files_int:
    files.append(int_path / file)
for file in files_ea:
    files.append(ea_path / file)

files

[PosixPath('../data/reporting/intelligence/int_abortion_o4mini_Ania.csv'),
 PosixPath('../data/reporting/intelligence/int_travel_o4mini_Maria.csv'),
 PosixPath('../data/reporting/intelligence/int_travel_o4mini_Marysia.csv'),
 PosixPath('../data/reporting/intelligence/int_ai_o4mini_Marysia.csv'),
 PosixPath('../data/reporting/intelligence/int_travel_o4mini_Ania.csv'),
 PosixPath('../data/reporting/intelligence/int_ai_o4mini_Ania.csv'),
 PosixPath('../data/reporting/intelligence/int_ai_o4mini_Maria.csv'),
 PosixPath('../data/reporting/intelligence/int_abortion_o4mini_Maria.csv'),
 PosixPath('../data/reporting/intelligence/int_abortion_o4mini_Marysia.csv'),
 PosixPath('../data/reporting/emotional_affect/pea_ai_o4mini_Maria.csv'),
 PosixPath('../data/reporting/emotional_affect/nea_abortion_o4mini_Maria.csv'),
 PosixPath('../data/reporting/emotional_affect/nea_ai_o4mini_Maria.csv'),
 PosixPath('../data/reporting/emotional_affect/nea_travel_o4mini_Ania.csv'),
 PosixPath('../data/reporting/em

In [15]:
data2 = pd.read_csv("../data/processed/processed_without_ai.csv")
data2.head()

,post_travel,post_abortion,adm,riv,gender,gender_3_text,age,ethnic_background,ethnic_background_8_text,education,...,twitter,none,facebook,instagram,tiktok,linkedin,pinterest,other,other_portals_7_text,post_ai
0,Just got back from a trip to Italy. It was eye...,That is crazy...,3.333,2.556,1.0,NaN,26,1.0,NaN,2.0,...,6.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,"AI can be scary, but used correctly it can be ..."
1,Travel can be extremely rewarding for both the...,This is complete government overreach. Infring...,2.556,1.889,1.0,NaN,34,1.0,NaN,2.0,...,2.0,NaN,1.0,1.0,NaN,1.0,1.0,NaN,NaN,"Like anything else, the ethics of tools are ho..."
2,Gettysburg was my favorite vacation so far. Th...,I don't even know what to say. This is fucked.,2.444,1.556,2.0,NaN,36,1.0,NaN,4.0,...,6.0,NaN,1.0,1.0,1.0,NaN,1.0,NaN,NaN,Having way too much fun having ChatGPT roast m...
3,Things went great on my trip last year and as ...,it is somehow unexpected thing that has occurs...,5.111,3.556,1.0,NaN,65,5.0,NaN,4.0,...,6.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,artificial intelligence has made things easier...
4,I believe the some of the most beautiful count...,I believe there should be just be better laws ...,3.778,1.889,2.0,NaN,20,7.0,NaN,4.0,...,1.0,NaN,1.0,1.0,NaN,1.0,1.0,1.0,"Twitter, Youtube",In the near future I see a mass involvement of...


In [36]:
data = pd.DataFrame(columns=[
    "intelligence_score",
    "positive_emotional_affect_score",
    "negative_emotional_affect_score",
    "post_type",
    "model",
    "role"
])

dfs = []

for file in files:
    df = pd.read_csv(file)
    m = re.search(r"([A-Za-z0-9\_]+).csv", str(file))
    match = re.findall(r"[A-Za-z0-9]+", m.group(0))
    if match[0] == "int":
        df["intelligence_score"] = df["score"].map(mapping_int)
    elif match[0] == "pea":
        df["positive_emotional_affect_score"] = df["score"].map(mapping_pos)
    elif match[0] == "nea":
        df["negative_emotional_affect_score"] = df["score"].map(mapping_neg)
    
    df["post_type"] = match[1]
    df["model"] = match[2]
    df["role"] = match[3]

    df.drop(columns=["score","reason"], inplace=True)

    print("post_"+match[1])
    df = df.merge(data2, how="left", left_on=["post"], right_on=["post_"+match[1]])

    df.drop(columns=["post_ai", "post_travel", "post_abortion"], inplace=True)


    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data = data.groupby(
    ["post", "post_type", "model", "role"], as_index=False
    ).first()

data.shape

post_abortion
post_travel
post_travel
post_ai
post_travel
post_ai
post_ai
post_abortion
post_abortion
post_ai
post_abortion
post_ai
post_travel
post_ai
post_abortion
post_travel
post_travel
post_abortion
post_ai
post_ai
post_travel
post_ai
post_abortion
post_abortion
post_travel
post_travel
post_abortion


(3969, 28)

In [37]:
data2[data2.post_abortion == "#BabiesLivesMatter"]

,post_travel,post_abortion,adm,riv,gender,gender_3_text,age,ethnic_background,ethnic_background_8_text,education,...,twitter,none,facebook,instagram,tiktok,linkedin,pinterest,other,other_portals_7_text,post_ai
62,Greatly enjoying the road trip with my sons cr...,#BabiesLivesMatter,3.222,2.111,1.0,NaN,47,1.0,NaN,4.0,...,2.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,Who’s ready for our AI overlords?


In [39]:
data.columns

Index(['post', 'post_type', 'model', 'role', 'intelligence_score', 'adm',
       'riv', 'gender', 'gender_3_text', 'age', 'ethnic_background',
       'ethnic_background_8_text', 'education', 'education_6_text',
       'employment', 'employment_7_text', 'marital_status', 'twitter', 'none',
       'facebook', 'instagram', 'tiktok', 'linkedin', 'pinterest', 'other',
       'other_portals_7_text', 'positive_emotional_affect_score',
       'negative_emotional_affect_score'],
      dtype='object')

In [7]:
data.to_csv("../data/processed/final_results.csv", index=False)

In [9]:
data2.head()

,post_travel,post_abortion,adm,riv,gender,gender_3_text,age,ethnic_background,ethnic_background_8_text,education,...,twitter,none,facebook,instagram,tiktok,linkedin,pinterest,other,other_portals_7_text,post_ai
0,Just got back from a trip to Italy. It was eye...,That is crazy...,3.333,2.556,1.0,NaN,26,1.0,NaN,2.0,...,6.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,"AI can be scary, but used correctly it can be ..."
1,Travel can be extremely rewarding for both the...,This is complete government overreach. Infring...,2.556,1.889,1.0,NaN,34,1.0,NaN,2.0,...,2.0,NaN,1.0,1.0,NaN,1.0,1.0,NaN,NaN,"Like anything else, the ethics of tools are ho..."
2,Gettysburg was my favorite vacation so far. Th...,I don't even know what to say. This is fucked.,2.444,1.556,2.0,NaN,36,1.0,NaN,4.0,...,6.0,NaN,1.0,1.0,1.0,NaN,1.0,NaN,NaN,Having way too much fun having ChatGPT roast m...
3,Things went great on my trip last year and as ...,it is somehow unexpected thing that has occurs...,5.111,3.556,1.0,NaN,65,5.0,NaN,4.0,...,6.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,artificial intelligence has made things easier...
4,I believe the some of the most beautiful count...,I believe there should be just be better laws ...,3.778,1.889,2.0,NaN,20,7.0,NaN,4.0,...,1.0,NaN,1.0,1.0,NaN,1.0,1.0,1.0,"Twitter, Youtube",In the near future I see a mass involvement of...
